# Reproduce all paper tables and figures

This notebook is a thin, annotated wrapper around [`reproduce.sh`](reproduce.sh) —
the **single source of truth** for the reproduction commands. `reproduce.sh full`
is a faithful, in-order transcription of the exact commands that produced the
published result grids (the original Colab campaign notebooks).

Set `QUICK = True` for a fast end-to-end toolchain check on a reduced grid
(minutes); set `QUICK = False` to reproduce the full paper grid (long-running —
many CPU-hours).

> **Backend note.** The multi-dimensional-knapsack decisions are differentiated
> with the `cvxpylayers` conic layer (open SCS/ECOS solvers, no MOSEK required);
> evaluation uses exact conic solves.


In [ ]:
# QUICK = True  -> toolchain check only. Runs 2 methods (FDFL, FPTO) on a tiny
#                  grid: healthcare m=800/N_train<=6/40 steps, knapsack
#                  m=30/N_train=4/8 steps. The numbers it prints are NOT the
#                  paper's and are not meant to be compared against them.
# QUICK = False -> the full published grid (many CPU-hours).
QUICK = True

print(("!" * 70) + "\n" + (
    "QUICK=True: toolchain check on a tiny grid (2 methods, reduced m/N/steps).\n"
    "Results will NOT match the paper. Set QUICK=False to reproduce the paper."
    if QUICK else
    "QUICK=False: full paper reproduction. This takes many CPU-hours."
) + "\n" + ("!" * 70))


## 1. Install dependencies

Builds the bundled `fair_dfl` algorithm source (`fair_dfl_moo/`) and installs the
differentiable-layer stack at pinned versions (open solvers only — no commercial
licenses required).


In [ ]:
%pip install -q -r requirements.txt


## 2. Build the dataset

Downloads the public Obermeyer et al. (2019) patient sample (not redistributed
here) and reconstructs the processed cohort used by the experiments.


In [ ]:
!python data/prepare_data.py


## 3. Run the pipeline

`smoke` = tuning + finals for two methods on a tiny grid; `full` = the complete
published grid (healthcare capacity/α/fairness/N axes; knapsack capacity ×
imbalance, K=4), then aggregation, tables, and figures.

> One supplement table (`tab:supp-md-misspec`, the *preliminary* predictor-class
> ladder) comes from a separate per-configuration HP-search ablation that is out
> of scope for this package; it is skipped automatically.


In [ ]:
import subprocess, sys
mode = 'smoke' if QUICK else 'full'
rc = subprocess.call(['bash', 'reproduce.sh', mode])
print('reproduce.sh', mode, '->', 'OK' if rc == 0 else f'FAILED (exit {rc})')
sys.exit(rc) if rc else None


## 4. Outputs

- main-text table → `tables/tab_sec53_anchor.tex`
- main-text figures → `figures/fig_sec53_*.{pdf,png}`
- online-supplement tables → `tables/supplement/supp_*.tex`
- online-supplement figures → `figures/supplement/`


In [ ]:
!find tables figures -type f | sort
